# Personal Reading Tracker
### A Python + Tableau project analyzing my reading habits

This notebook takes a raw export from my reading app (Fable, exported in a Goodreads-style
CSV format), cleans it, enriches it with genre and page-count data from the free **Open
Library API**, and produces a polished CSV ready to visualize in **Tableau**.

**Pipeline overview:**
1. Load and clean the raw export
2. Filter to books actually finished, and parse read dates
3. Enrich each book with genre + page count via Open Library
4. Export a final, Tableau-ready dataset + a pre-aggregated monthly summary

**A note on data quality:** this is real, messy, real-world data — not a tidy tutorial
dataset. Some books are missing read dates, some ISBNs are invalid, and not every title
will be found on Open Library. Part of this project is handling those gaps honestly rather
than papering over them, which I think matters more in a portfolio piece than a perfect-looking
dataset would.


## Step 1: Load and Clean the Raw Export

The raw file is `goodreads_import.csv`, exported from Fable via a third-party tool that
converts Fable's library into a Goodreads-style CSV.

Three cleanup problems to solve right away:
- **ISBNs are wrapped** in Excel's formula syntax, e.g. `="9780316592253"`
- **Some "ISBNs" aren't real ISBNs** — they're Fable's internal book IDs (e.g. `QZRamyEMDh`),
  which won't work for an Open Library lookup
- **Date Read isn't always filled in** — only books I logged a finish date for can be plotted
  on a monthly timeline


In [2]:
import csv
import re
from datetime import datetime

INPUT_FILE = "goodreads_import.csv"


def clean_isbn(raw_isbn: str) -> str:
    """Strip the ="..." Excel wrapper Goodreads/Fable exports wrap ISBNs in."""
    if not raw_isbn:
        return ""
    match = re.search(r'"([^"]+)"', raw_isbn)
    if match:
        return match.group(1).strip()
    return raw_isbn.strip().strip('="')


def is_valid_isbn(isbn: str) -> bool:
    """
    A real ISBN is 10 or 13 digits (the last character of ISBN-10 can be 'X').
    Fable's internal IDs (e.g. 'QZRamyEMDh') are alphanumeric and won't match.
    """
    if not isbn:
        return False
    cleaned = isbn.replace("-", "").strip()
    if len(cleaned) == 10:
        return bool(re.match(r'^\d{9}[\dX]$', cleaned))
    if len(cleaned) == 13:
        return cleaned.isdigit()
    return False


def parse_date_read(raw_date: str):
    """Fable exports dates as YYYY/MM/DD. Returns a date object or None."""
    if not raw_date or not raw_date.strip():
        return None
    try:
        return datetime.strptime(raw_date.strip(), "%Y/%m/%d").date()
    except ValueError:
        return None


with open(INPUT_FILE, newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    rows = list(reader)

print(f"Total rows in raw export: {len(rows)}")


Total rows in raw export: 322


### Filter to "read" books and clean each row

Fable's export includes books on every shelf (`read`, `to-read`, `currently-reading`).
This project only cares about books I've actually finished.


In [3]:
read_books = [r for r in rows if r["Exclusive Shelf"] == "read"]
print(f"Books on the 'read' shelf: {len(read_books)}")

cleaned_rows = []
valid_isbn_count = 0
dated_count = 0

for r in read_books:
    isbn = clean_isbn(r["ISBN"])
    valid_isbn = is_valid_isbn(isbn)
    if valid_isbn:
        valid_isbn_count += 1

    date_read = parse_date_read(r["Date Read"])
    if date_read:
        dated_count += 1

    cleaned_rows.append({
        "title": r["Title"].strip(),
        "author": r["Author"].strip(),
        "isbn": isbn,
        "isbn_valid": valid_isbn,
        "my_rating": r["My Rating"].strip() or "0",
        "date_read": date_read.isoformat() if date_read else "",
        "read_year": date_read.year if date_read else "",
        "read_month": date_read.month if date_read else "",
        "read_month_label": date_read.strftime("%Y-%m") if date_read else "",
        "has_date": bool(date_read),
        "year_published": r["Year Published"].strip(),
        # placeholders filled in during enrichment, below
        "genre": "",
        "page_count": "",
    })

print(f"Books with a valid, lookup-able ISBN: {valid_isbn_count}")
print(f"Books with a parsed Date Read: {dated_count}")


Books on the 'read' shelf: 233
Books with a valid, lookup-able ISBN: 199
Books with a parsed Date Read: 97


### Save the cleaned dataset

This checkpoint file means I don't have to re-parse the raw export every time —
later steps (and re-runs) start from here.


In [4]:
CLEAN_FILE = "clean_books.csv"

fieldnames = [
    "title", "author", "isbn", "isbn_valid", "my_rating",
    "date_read", "read_year", "read_month", "read_month_label",
    "has_date", "year_published", "genre", "page_count",
]

with open(CLEAN_FILE, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(cleaned_rows)

print(f"Wrote {len(cleaned_rows)} rows to {CLEAN_FILE}")


Wrote 233 rows to clean_books.csv


## Step 2: Enrich with Genre & Page Count (Open Library API)

Fable's export doesn't include genre or page count, so this step calls the free
**[Open Library API](https://openlibrary.org/developers/api)** (no key required) to fill
in both fields for each book.

**Lookup strategy:**
- If a book has a **valid ISBN**, look it up directly via the ISBN endpoint
- If not (internal Fable ID, or the ISBN lookup comes back empty), **fall back** to a
  title + author search

**⚠️ This cell requires an internet connection.** It also runs a deliberate short delay
between requests to avoid hammering Open Library's free, rate-limited API — expect this
cell to take a few minutes on the full dataset.


In [5]:
import time
import requests

ENRICHED_FILE = "enriched_books.csv"
REQUEST_DELAY_SECONDS = 0.5
HEADERS = {"User-Agent": "PersonalReadingTracker/1.0 (portfolio project)"}


def lookup_by_isbn(isbn: str):
    """Try to fetch genre + page count using the ISBN endpoint."""
    url = f"https://openlibrary.org/isbn/{isbn}.json"
    try:
        resp = requests.get(url, headers=HEADERS, timeout=10)
        if resp.status_code != 200:
            return None, None
        data = resp.json()
        page_count = data.get("number_of_pages")
        subjects = data.get("subjects", [])
        genre = subjects[0] if subjects else None
        return genre, page_count
    except requests.RequestException:
        return None, None


def lookup_by_title_author(title: str, author: str):
    """Fallback search when there's no usable ISBN."""
    url = "https://openlibrary.org/search.json"
    params = {"title": title, "author": author.split(",")[0].strip(), "limit": 1}
    try:
        resp = requests.get(url, headers=HEADERS, params=params, timeout=10)
        if resp.status_code != 200:
            return None, None
        data = resp.json()
        docs = data.get("docs", [])
        if not docs:
            return None, None
        doc = docs[0]
        page_count = doc.get("number_of_pages_median")
        subjects = doc.get("subject", [])
        genre = subjects[0] if subjects else None
        return genre, page_count
    except requests.RequestException:
        return None, None


### Run the enrichment loop

This re-loads `clean_books.csv` (so this section can be re-run independently of Step 1)
and calls Open Library for every book.


In [6]:
with open(CLEAN_FILE, newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    enrich_rows = list(reader)

print(f"Enriching {len(enrich_rows)} books from Open Library...")
print("This will take a few minutes due to API rate limiting.\n")

found_count = 0

for i, row in enumerate(enrich_rows, start=1):
    genre, pages = None, None

    if row["isbn_valid"] == "True" and row["isbn"]:
        genre, pages = lookup_by_isbn(row["isbn"])

    if genre is None and pages is None:
        genre, pages = lookup_by_title_author(row["title"], row["author"])

    row["genre"] = genre or "Unknown"
    row["page_count"] = pages if pages else ""

    if genre or pages:
        found_count += 1

    print(f"[{i}/{len(enrich_rows)}] {row['title'][:50]:50s} -> genre={genre}, pages={pages}")

    time.sleep(REQUEST_DELAY_SECONDS)

print(f"\nDone. Found data for {found_count}/{len(enrich_rows)} books.")


Enriching 233 books from Open Library...
This will take a few minutes due to API rate limiting.

[1/233] Midnight Sun                                       -> genre=None, pages=672
[2/233] The Devils                                         -> genre=None, pages=576
[3/233] Brigands & Breadknives                             -> genre=None, pages=352
[4/233] Alchemised                                         -> genre=None, pages=1040
[5/233] Earthflown                                         -> genre=None, pages=None
[6/233] The Vanishing Cherry Blossom Bookshop              -> genre=None, pages=None
[7/233] Fun Home                                           -> genre=None, pages=None
[8/233] The Elsewhere Express                              -> genre=None, pages=432
[9/233] Interview with the Vampire                         -> genre=None, pages=None
[10/233] Nine Goblins                                       -> genre=None, pages=None
[11/233] The Unworthy                                   

In [7]:
with open(ENRICHED_FILE, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(enrich_rows[0].keys()))
    writer.writeheader()
    writer.writerows(enrich_rows)

print(f"Wrote enriched data to {ENRICHED_FILE}")


Wrote enriched data to enriched_books.csv


## Step 3: Build the Final Tableau-Ready Dataset

The last step shapes the enriched data into the final file I'll import into Tableau, plus
a smaller pre-aggregated **monthly summary** table (books read and pages read per month),
so I can build a quick timeline chart without writing calculated fields inside Tableau.

Two small additions here:
- `rating_label` — turns my numeric 0–5 rating into a readable label
- `has_genre` / `has_pages` — boolean flags so I can filter out "Unknown" values cleanly
  when building Tableau charts


In [8]:
from collections import defaultdict

TABLEAU_FILE = "reading_tracker_tableau.csv"
SUMMARY_FILE = "monthly_summary.csv"


def rating_label(rating: str) -> str:
    mapping = {
        "0": "Not rated", "1": "1 star", "2": "2 stars",
        "3": "3 stars", "4": "4 stars", "5": "5 stars",
    }
    return mapping.get(rating.strip(), "Not rated")


with open(ENRICHED_FILE, newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    final_input_rows = list(reader)

print(f"Preparing {len(final_input_rows)} books for Tableau...")

final_rows = []
for r in final_input_rows:
    pages_raw = r.get("page_count", "").strip()
    has_pages = pages_raw.isdigit() and int(pages_raw) > 0
    genre = r.get("genre", "Unknown").strip() or "Unknown"

    final_rows.append({
        "title": r["title"],
        "author": r["author"],
        "isbn": r["isbn"],
        "my_rating": r["my_rating"],
        "rating_label": rating_label(r["my_rating"]),
        "date_read": r["date_read"],
        "read_year": r["read_year"],
        "read_month": r["read_month"],
        "read_month_label": r["read_month_label"],
        "has_date": r["has_date"],
        "genre": genre,
        "has_genre": genre != "Unknown",
        "page_count": pages_raw if has_pages else "",
        "has_pages": has_pages,
    })

with open(TABLEAU_FILE, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(final_rows[0].keys()))
    writer.writeheader()
    writer.writerows(final_rows)

print(f"Wrote {TABLEAU_FILE} -- import this into Tableau.")


Preparing 233 books for Tableau...
Wrote reading_tracker_tableau.csv -- import this into Tableau.


In [ ]:
monthly = defaultdict(lambda: {"books": 0, "pages": 0})
for r in final_rows:
    if r["has_date"] == "True" and r["read_month_label"]:
        key = r["read_month_label"]
        monthly[key]["books"] += 1
        if r["has_pages"]:
            monthly[key]["pages"] += int(r["page_count"])

with open(SUMMARY_FILE, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["month", "books_read", "total_pages"])
    for month in sorted(monthly.keys()):
        writer.writerow([month, monthly[month]["books"], monthly[month]["pages"]])

print(f"Wrote {SUMMARY_FILE} -- a pre-aggregated monthly view, optional but handy.")
print("\nAll done! Open reading_tracker_tableau.csv in Tableau to start building the dashboard.")


## Quick Preview

A fast sanity check before jumping into Tableau — a peek at the final table and a simple
bar chart of books read per month, right here in the notebook.


In [9]:
import pandas as pd

df = pd.read_csv(TABLEAU_FILE)
df.head()


,title,author,isbn,my_rating,rating_label,date_read,read_year,read_month,read_month_label,has_date,genre,has_genre,page_count,has_pages
0,Midnight Sun,Stephenie Meyer,9780316592253,2,2 stars,2026-06-15,2026.0,6.0,2026-06,True,Unknown,False,672.0,True
1,The Devils,Joe Abercrombie,9781250880062,5,5 stars,2026-06-11,2026.0,6.0,2026-06,True,Unknown,False,576.0,True
2,Brigands & Breadknives,Travis Baldree,9781250334893,4,4 stars,2026-05-31,2026.0,5.0,2026-05,True,Unknown,False,352.0,True
3,Alchemised,SenLinYu,9780593972700,4,4 stars,2026-05-21,2026.0,5.0,2026-05,True,Unknown,False,1040.0,True
4,Earthflown,Frances Wren,QZRamyEMDh,3,3 stars,2026-05-10,2026.0,5.0,2026-05,True,Unknown,False,NaN,False


In [10]:
import matplotlib.pyplot as plt

summary_df = pd.read_csv(SUMMARY_FILE)

plt.figure(figsize=(10, 4))
plt.bar(summary_df["month"], summary_df["books_read"])
plt.xticks(rotation=45, ha="right")
plt.ylabel("Books Read")
plt.title("Books Read Per Month")
plt.tight_layout()
plt.show()


FileNotFoundError: [Errno 2] No such file or directory: 'monthly_summary.csv'

## Next Step: Tableau

With `reading_tracker_tableau.csv` ready, the next step is building out the dashboard in
Tableau:

1. **Books & pages per month** — using `monthly_summary.csv`
2. **Genre breakdown** — treemap or pie of the `genre` column
3. **Ratings distribution** — bar chart of `rating_label`
4. **Genre vs. rating** — does any genre consistently get rated higher?

## Known Data Limitations

Worth noting honestly, both here and in any interview conversation about this project:

- Only books with a confirmed `Date Read` appear in the monthly timeline — books read but
  not dated are still included in genre/rating analysis, just not the time series.
- A handful of books had non-standard ISBNs (Fable-internal IDs rather than real ISBNs)
  and relied on the title/author search fallback, which is slightly less reliable.
- Open Library's genre tagging is community-sourced, so some books come back as "Unknown."
